### Deep Q-learning for multi-armed bandits (PyTorch)

This notebook runs a "multi-armed bandit" problem using the DQN framework. This is definitely using a hammer for a nail, but it illustrates the approach to more complex deep Q-learning problems.

For multi-armed bandits, the
optimal strategy is to always pull the arm with the largest expected reward, as soon
as you know that's the best arm.

This notebook also serves as a working example for automatic differentiation in PyTorch.

Adapted from [this post](https://towardsdatascience.com/a-minimal-working-example-for-deep-q-learning-in-tensorflow-2-0-e0ca8a944d5e) (originally TensorFlow, converted here to PyTorch).

**Running on Google Colab with a GPU:** In the Colab menu, go to `Runtime > Change runtime type` and set `Hardware accelerator` to `GPU` (T4 is fine), then run all cells. The code below automatically detects and uses the GPU if one is available.

In [ ]:
# Colab already ships with PyTorch preinstalled, but this is harmless if it's already there.
%pip install -q torch matplotlib numpy

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from time import sleep
from IPython.display import clear_output
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
def get_reward(bandit: float, device: torch.device):
    """Generate reward for selected bandit"""
    mean = torch.tensor(bandit, dtype=torch.float32, device=device)
    std = torch.tensor(1.0, dtype=torch.float32, device=device)
    reward = torch.normal(mean=mean, std=std)
    return reward


class QNetwork(nn.Module):
    """Q-network with q-values per action as output"""

    def __init__(self, state_dim: int, action_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 10),
            nn.ReLU(),
            nn.Linear(10, 10),
            nn.ReLU(),
            nn.Linear(10, 10),
            nn.ReLU(),
            nn.Linear(10, action_dim),
        )

    def forward(self, x):
        return self.net(x)


def mean_squared_error_loss(q_value: torch.Tensor, reward: torch.Tensor):
    """Compute mean squared error loss"""
    loss = 0.5 * (q_value - reward) ** 2
    return loss


def plot(q_values: torch.Tensor, bandits: np.ndarray, episode):
    """Plot bar chart with selection probability per bandit"""
    q_values_np = q_values.detach().cpu().numpy()
    q_values_plot = [q_values_np[i] for i in range(len(q_values_np))]
    bandit_plot = [bandits[i] for i in range(len(bandits))]
    width = 0.4
    x = np.arange(len(bandits))

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.bar(x - width / 2, q_values_plot, width, label="Q-values")
    ax.bar(x + width / 2, bandit_plot, width, label="True values")

    ax.set_xticks(range(len(q_values_np)))
    plt.xlabel("Bandit")
    plt.ylabel("Value")
    plt.legend(loc="best")
    plt.title('Episode: %d' % episode)
    plt.show()
    return

In [ ]:
# Initialize parameters
state = torch.tensor([[1.0]], device=device)
bandits = np.array([0.9, 1.2, 0.7, 1.0, 1.5])
state_dim = state.shape[1]
action_dim = len(bandits)
exploration_rate = 0.2
learning_rate = 0.01
num_episodes = 1000

# Construct Q-network
q_network = QNetwork(state_dim, action_dim).to(device)

# Define optimizer
opt = optim.Adam(q_network.parameters(), lr=learning_rate)

for i in range(num_episodes + 1):
    # Obtain Q-values from network
    q_values = q_network(state)

    epsilon = np.random.rand()
    if epsilon <= exploration_rate:
        # Select random action
        action = np.random.choice(len(bandits))
    else:
        # Select action with highest q-value
        action = torch.argmax(q_values).item()

    # Obtain reward from bandit
    reward = get_reward(bandits[action], device)

    # Obtain Q-value
    q_value = q_values[0, action]

    # Compute loss value
    loss_value = mean_squared_error_loss(q_value, reward)

    # Compute gradients and apply them to update network weights
    opt.zero_grad()
    loss_value.backward()
    opt.step()

    # Print console output
    if np.mod(i, int(10 * np.log(i + 1) / 2) + 1) == 0:
        clear_output(wait=True)
        print("\n======episode", i, "======")
        q_values_row = q_values[0].detach().cpu().numpy()
        print("Q-values", ["%.3f" % n for n in q_values_row])
        print("Deviation",
              ["%.1f%%" % float(100 * (q_values_row[j] - bandits[j]) / bandits[j]) for j in range(len(q_values_row))])
        plot(q_values[0], bandits, episode=i)
        sleep(1)